# 02 — Bronze ingestion

**Purpose:** Load the landed Kitsune SYN DoS files into Delta tables in the `bronze` schema, unchanged, plus ingestion metadata.

**Reads from:** `/Volumes/kitsune_dev/raw/landing/`
- `SYN DoS_labels.csv.gz`
- `SYN DoS_dataset.csv.gz`

**Writes to:**
- `kitsune_dev.bronze.syn_dos_labels_raw`: labels file as-is (`_c0` row number, `x` label)
- `kitsune_dev.bronze.syn_dos_features_raw`: 115 feature columns (`_c0` to `_c114`), plus `_row_number` (1-based position in the file)

All columns stay as strings; typing and cleaning happen in silver. Both tables get `_source_file` and `_ingested_at`, plus table/column comments and tags.

**Validates:**
- The features file has 115 columns
- Features and labels have the same number of rows
- `_row_number` goes from 1 to n with no gaps

If any check fails, the notebook raises an error and stops.

**When to run:** After `01_land_source_files`.

**Idempotent:** Yes. Tables are written with `overwrite` (full reload), so re-running produces the same result. Comments and tags are re-applied after each write.

**Runtime:** ~4 min. The features file is a ~2 GB gzip, which is not splittable, so Spark reads it with a single task.

In [0]:
from pyspark.sql import functions as F

# Landing zone and source files
LANDING_DIR = "/Volumes/kitsune_dev/raw/landing/"
LABELS_PATH = LANDING_DIR + "SYN DoS_labels.csv.gz"
DATASET_PATH = LANDING_DIR + "SYN DoS_dataset.csv.gz"

# Bronze tables
FEATURES_TABLE = "kitsune_dev.bronze.syn_dos_features_raw"
LABELS_TABLE = "kitsune_dev.bronze.syn_dos_labels_raw"

# Expected number of feature columns in the dataset file
EXPECTED_FEATURE_COLUMNS = 115

In [0]:
# Read the labels file (has a header row)
labels_df = spark.read.csv(LABELS_PATH, header=True)

# Add ingestion metadata to every row
labels_bronze_df = (
    labels_df                                                   
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)                                                              

# Write labels to bronze as a managed Delta table
(
    labels_bronze_df                                          
    .write
    .mode("overwrite")
    .saveAsTable(LABELS_TABLE)                                
)

In [0]:
%sql
-- Table comment
COMMENT ON TABLE kitsune_dev.bronze.syn_dos_labels_raw IS 'Raw labels for the Kitsune SYN DoS capture, one row per packet in chronological order. Loaded as-is from SYN DoS_labels.csv.gz (all strings). Not cleaned.';

-- Column comments
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw ALTER COLUMN _c0 COMMENT 'Row number from the source file (1-based), likely added when the file was exported. Used in silver to align each label with its feature row.';
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw ALTER COLUMN x COMMENT 'Packet label: 1 = malicious (attack), 0 = benign. Renamed to label in silver.';
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw ALTER COLUMN _source_file COMMENT 'Path of the source file this row was read from, filled by Spark (_metadata.file_path).';
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw ALTER COLUMN _ingested_at COMMENT 'Timestamp (UTC) when the row was loaded into bronze.';

-- Add tags to the tables
ALTER TABLE kitsune_dev.bronze.syn_dos_labels_raw
SET TAGS ('layer' = 'bronze', 'source' = 'uci_kitsune', 'dataset' = 'syn_dos', 'contains_pii' = 'false');

In [0]:
# Read the features file (no header, ~2 GB)
features_df = spark.read.csv(DATASET_PATH, header=False)

# The features file must have the expected number of columns
actual_columns = len(features_df.columns)
if actual_columns != EXPECTED_FEATURE_COLUMNS:
    raise ValueError(f"Expected {EXPECTED_FEATURE_COLUMNS} columns, found {actual_columns}")

# Add row position and ingestion metadata
features_bronze_df = (
    features_df
    .withColumn("_row_number", F.monotonically_increasing_id() + 1)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# Write features to bronze as a managed Delta table
(
    features_bronze_df
    .write
    .mode("overwrite")
    .saveAsTable(FEATURES_TABLE)
)

In [0]:
%sql
-- Table comment
COMMENT ON TABLE kitsune_dev.bronze.syn_dos_features_raw IS 'Raw features for the Kitsune SYN DoS capture, one row per packet in chronological order. Columns _c0 to _c114 are the 115 traffic statistics in their original order (all strings, renamed in silver). Loaded as-is from SYN DoS_dataset.csv.gz. Not cleaned.';

-- Column comments
ALTER TABLE kitsune_dev.bronze.syn_dos_features_raw ALTER COLUMN _row_number COMMENT 'Row position in the source file (1-based), added by the pipeline at ingestion. Matches _c0 in syn_dos_labels_raw and is used in silver to join features with labels.';
ALTER TABLE kitsune_dev.bronze.syn_dos_features_raw ALTER COLUMN _source_file COMMENT 'Path of the source file this row was read from, filled by Spark (_metadata.file_path).';
ALTER TABLE kitsune_dev.bronze.syn_dos_features_raw ALTER COLUMN _ingested_at COMMENT 'Timestamp (UTC) when the row was loaded into bronze.';

-- Table tags
ALTER TABLE kitsune_dev.bronze.syn_dos_features_raw
SET TAGS ('layer' = 'bronze', 'source' = 'uci_kitsune', 'dataset' = 'syn_dos', 'contains_pii' = 'false');

In [0]:
# Row count and row_number range of the features table
features_stats = (
    spark.table(FEATURES_TABLE)
    .agg(
        F.count("*").alias("n"),
        F.min("_row_number").alias("min_row"),
        F.max("_row_number").alias("max_row"),
    )
    .first()
)

labels_count = spark.table(LABELS_TABLE).count()

# 1. Features and labels must have the same number of rows
if features_stats["n"] != labels_count:
    raise ValueError(f"Row count mismatch: features={features_stats['n']}, labels={labels_count}")

# 2. Row numbers must go from 1 to n with no gaps
if features_stats["min_row"] != 1 or features_stats["max_row"] != features_stats["n"]:
    raise ValueError(f"_row_number is not contiguous: min={features_stats['min_row']}, max={features_stats['max_row']}, n={features_stats['n']}")

print("Bronze validations passed")

Bronze validations passed


## Validate label row numbers

**What:** check that `_c0` in the labels table goes from 1 to n, with no gaps and no repeats.

**Why:** `_c0` is the key silver uses to join each label with its packet. A missing number and a repeated one would cancel out in the join's row count and pass silently, mislabeling packets.

**Result:** "Label row numbers OK" or a `ValueError`.

In [0]:
# Stats of the label row numbers (one pass over the table)
labels_stats = (
    spark.table(LABELS_TABLE)
    .select(F.col("_c0").cast("bigint").alias("row_num"))
    .agg(
        F.count("*").alias("n"),
        F.count_distinct("row_num").alias("n_distinct"),
        F.min("row_num").alias("min_row"),
        F.max("row_num").alias("max_row"),
    )
    .first()
)

# Row numbers must be exactly 1..n: no repeats, starts at 1, ends at n
if (
    labels_stats["n_distinct"] != labels_stats["n"]
    or labels_stats["min_row"] != 1
    or labels_stats["max_row"] != labels_stats["n"]
):
    raise ValueError(f"Label row numbers are not 1..n: {labels_stats}")

print("Label row numbers OK")

**Next:** `03_silver_transformation`, which casts types, checks for nulls/NaN and joins features with labels.